# Estimating the interferometer residual OPD from frequency-modulated phasemeter data

This notebook documents, end to end, how the `het_ifo_opd` package estimates the
**residual optical path-length difference (OPD)** of the FM1 interferometer from
Moku:Pro Phasemeter recordings, and then runs the full FM1 analysis and
discusses the results.

**Contents**

1. [The measurement principle](#principle) — how a laser-frequency modulation encodes the OPD.
2. [The data](#data) — what a Moku:Pro Phasemeter file contains.
3. [Why the *differential* phase](#differential) — common-mode rejection of laser noise.
4. [The estimator](#estimator) — the leakage problem, complex-baseband demodulation, and coherence-based integration.
5. [Worked single-file example](#single) — a clean acquisition, step by step.
6. [Full FM1 analysis](#full) — every acquisition, with the correct modulation frequency per day.
7. [Discussion of results](#discussion).
8. [Conclusions](#conclusions).

In [ ]:
%matplotlib inline
import os
import logging

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from het_ifo_opd import OPDConfig, estimate_opd, estimate_opd_dataset, load_phasemeter
from het_ifo_opd.physics import phase_cycles_to_opd
from het_ifo_opd.estimators import demodulate, lockin_amplitude, single_bin_amplitude
from het_ifo_opd.plotting import plot_diagnostics, plot_dataset_summary
from speckit import compute_spectrum

logging.getLogger("het_ifo_opd").setLevel(logging.WARNING)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Locate the FM1 data directory (works whether the notebook is run from the
# repo root or from notebooks/).
_candidates = ["data/FM1", "../data/FM1"]
FM1_DIR = next((p for p in _candidates if os.path.isdir(p)), "FM1")
print("FM1 directory:", os.path.abspath(FM1_DIR))

# Default physics configuration (matches the FM1 set-up).
cfg = OPDConfig()
print(f"Laser wavelength       : {cfg.laser_wavelength*1e9:.0f} nm "
      f"(carrier {cfg.laser_frequency/1e12:.2f} THz)")
print(f"Actuator transfer fn   : {cfg.actuator_tf/1e6:.0f} MHz/V")
print(f"Modulation amplitude   : {cfg.mod_vpp:.1f} Vpp")
print(f"=> peak freq deviation : {cfg.freq_dev_peak/1e6:.0f} MHz (zero-to-peak)")


<a id="principle"></a>
## 1. The measurement principle

An interferometer with an optical path-length difference $\mathrm{OPD}$ between
its two arms converts the instantaneous **optical frequency** $\nu(t)$ of the
laser into an interferometric phase

$$
\varphi(t) \;=\; \nu(t)\,\frac{\mathrm{OPD}}{c}\qquad\text{[cycles]} .
$$

(Equivalently $\varphi = \mathrm{OPD}/\lambda$, since $\nu = c/\lambda$.) The
phasemeter reports $\varphi$ directly, in cycles.

If we now **modulate the laser frequency** by a small amount $\delta\nu(t)$ about
its mean $\nu_0$, the phase acquires a corresponding modulation

$$
\delta\varphi(t) \;=\; \frac{\mathrm{OPD}}{c}\,\delta\nu(t),
$$

because $\mathrm{d}\varphi/\mathrm{d}\nu = \mathrm{OPD}/c$. The relation is
**linear**: every Fourier component of $\delta\nu$ maps to the *same* component
of $\delta\varphi$, scaled by the constant $\mathrm{OPD}/c$.

Driving the laser-frequency actuator with a single tone at $f_\mathrm{mod}$,

$$
\delta\nu(t) = \delta\nu_\mathrm{pk}\cos(2\pi f_\mathrm{mod} t),
$$

therefore writes a phase tone of amplitude

$$
A_\varphi = \frac{\mathrm{OPD}}{c}\,\delta\nu_\mathrm{pk}
\qquad\Longrightarrow\qquad
\boxed{\;\mathrm{OPD} = \dfrac{A_\varphi\, c}{\delta\nu_\mathrm{pk}}\;}
$$

The peak frequency deviation follows from the known drive and actuator gain,

$$
\delta\nu_\mathrm{pk} = \frac{\mathrm{d}\nu}{\mathrm{d}V}\cdot\frac{V_\mathrm{pp}}{2}
= 376\ \mathrm{MHz/V}\times\frac{1\ \mathrm{V_{pp}}}{2} = 188\ \mathrm{MHz},
$$

(the amplitude is half of the peak-to-peak drive).

**Two things make this method attractive:**

* It is **wavelength-independent** — the OPD comes from the *frequency*
  modulation alone; the 1550 nm carrier is needed only for context.
* It needs only the **amplitude of a single tone**, which can be estimated
  optimally; the large, slowly-drifting DC interferometer phase is irrelevant.

So the entire problem reduces to: *measure the amplitude $A_\varphi$ of the
$f_\mathrm{mod}$ tone in the interferometric phase, as accurately as possible.*

<a id="data"></a>
## 2. The data

Each FM1 acquisition is a Moku:Pro Phasemeter file (`.zip`/`.csv`) containing
**two PLL channels** (A and B), each reporting set-frequency, tracked frequency,
phase (in cycles), and the in-phase/quadrature terms, sampled at $f_s\approx
596\ \mathrm{Hz}$ for $\sim600\ \mathrm{s}$.

We load a single anchored acquisition and look at the two channel phases.

In [ ]:
example_file = os.path.join(FM1_DIR,
    "FM1Day06_AnchoredAirOPD_EDUbase_R1_20260609_135823.zip")

data = load_phasemeter(example_file)
print(f"name      : {data.name}")
print(f"fs        : {data.fs:.4f} Hz")
print(f"duration  : {data.duration:.1f} s  ({data.t.size} samples)")
print(f"channels  : {data.nchan}")

a = data.channel_cycles[0]
b = data.channel_cycles[1]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(data.t, a - a.mean(), lw=0.5, label="ch A")
ax[0].plot(data.t, b - b.mean(), lw=0.5, label="ch B", alpha=0.7)
ax[0].set(xlabel="Time [s]", ylabel="Phase [cyc]",
          title="Each channel: dominated by common laser noise")
ax[0].legend()

ax[1].plot(data.t, (a - b) - (a - b).mean(), lw=0.5, color="C2")
ax[1].set(xlabel="Time [s]", ylabel="Phase [cyc]",
          title="Differential A-B: laser noise cancels")
fig.tight_layout()

print(f"\nstd(A)     = {np.std(a):.4e} cyc")
print(f"std(B)     = {np.std(b):.4e} cyc")
print(f"std(A-B)   = {np.std(a-b):.4e} cyc  "
      f"-> common-mode rejection ~ {np.std(a)/np.std(a-b):.0f}x")


<a id="differential"></a>
## 3. Why the *differential* phase

Both channels share the same laser, so each individual phase is dominated by the
laser's frequency noise coupling through its (large, ~cm-scale) absolute arm
imbalance — this is the huge, slow wander seen above. Their **difference**
$\varphi = \varphi_A - \varphi_B$ cancels this common contribution
(~$10^{3}$&times; rejection here) and leaves the **residual differential OPD** —
exactly the quantity of interest for a nominally balanced two-beam
interferometer.

The injected $f_\mathrm{mod}$ tone is common to both channels too, so it is
*also* strongly rejected in the difference: what survives is the tone amplitude
due to the **residual** OPD mismatch. The spectra below make this concrete — the
tone is huge in A and B, and reduced (but cleanly measurable) in A&minus;B.

`het_ifo_opd` therefore uses the differential phase as the observable, while still
reporting each channel's absolute OPD as a diagnostic.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for sig, lab, c in [(a, "ch A", "C0"), (b, "ch B", "C1"),
                    (a - b, "A - B (differential)", "C2")]:
    s = compute_spectrum(sig - np.mean(sig), data.fs, win="kaiser", olap="default")
    ax.loglog(s.f, s.asd, lw=0.8, color=c, label=lab)
ax.axvline(100.0, color="C3", ls="--", lw=1, label="100 Hz modulation")
ax.set(xlabel="Frequency [Hz]", ylabel=r"ASD [cyc/$\sqrt{\mathrm{Hz}}$]",
       title="Amplitude spectral density: tone is common-mode, OPD residual survives in A-B")
ax.legend()
fig.tight_layout()

# Quantify the 100 Hz tone in each, and the residual OPD it implies.
for sig, lab in [(a, "A"), (b, "B"), (a - b, "A-B")]:
    amp = single_bin_amplitude(sig - np.mean(sig), data.fs, 100.0)
    opd = phase_cycles_to_opd(amp, cfg.freq_dev_peak)
    print(f"{lab:>4}: tone amplitude = {amp:.3e} cyc  ->  OPD = {opd*1e3:8.3f} mm")


<a id="estimator"></a>
## 4. The estimator

### 4.1 The trap a naive lock-in falls into: spectral leakage

The differential phase is *dominated* by a very large, slow wander — the
residual $\nu_0\,\mathrm{OPD}(t)/c$ term — which is **tens of cycles**
peak-to-peak, even though the modulation tone we want to measure is only
$\sim10^{-4}$ cycles. A rectangular-window least-squares fit (or plain DFT) at
$f_0$ has sidelobes that fall off only as $1/f$, so this enormous low-frequency
content **leaks** into the $f_0$ bin and biases the amplitude high. Because the
leakage scales as $\sim1/T$, a naive full-record lock-in can be biased by up to
$\sim2\times$, and short per-segment fits by $\sim10\times$ — the origin of the
long-standing puzzle where the "per-segment OPD" sat far above the full-record
number.

A polynomial detrend and a Kaiser-windowed fit help, but the clean, definitive
fix is to isolate a narrow band around the tone *before* measuring it.

### 4.2 Primary: complex-baseband demodulation

We heterodyne the tone to DC and low-pass to a narrow one-sided bandwidth $B$:

$$
z(t) \;=\; 2\,\mathrm{LPF}_B\!\big[\,\varphi(t)\,e^{-2\pi i f_0 t}\,\big],
\qquad |z(t)| = A_\varphi(t)\ \ \text{[cyc, zero-to-peak]} .
$$

The low-pass rejects everything except a band $[f_0-B,\,f_0+B]$, so the
out-of-band wander physically **cannot leak in**. This makes $|z(t)|$ the
*instantaneous* tone amplitude — hence the instantaneous
$\mathrm{OPD}(t)=|z(t)|\,c/\delta\nu_\mathrm{pk}$ — while $\arg z(t)$ is the true
tone phase (its slope is any residual frequency offset). The instantaneous
$\mathrm{OPD}(t)$ is the real deliverable for records whose OPD physically
evolves.

The tone frequency $f_0$ is taken from the known drive (100 Hz on Day06, 95 Hz
on Day09) and refined on a fine periodogram grid.

### 4.3 Coherent vs incoherent integration (chosen automatically)

From $z(t)$ we form a **coherence**
$\rho = |\langle z\rangle| / \sqrt{\langle |z|^2\rangle}$ (after removing a
constant frequency offset), and let it decide how to collapse $z(t)$ to one
number:

* $\rho \ge$ `coherence_threshold` → the tone is phase-coherent; report the
  **coherent** amplitude $|\langle z\rangle|$ (optimal SNR).
* otherwise → the tone genuinely wanders in phase; report the noise-debiased
  **incoherent** amplitude
  $\sqrt{\langle|z|^2\rangle - \langle|z_\mathrm{off}|^2\rangle}$, where
  $z_\mathrm{off}$ is an off-tone demodulation that measures the in-band noise.

### 4.4 Cross-checks and uncertainty

* The **rectangular lock-in** is kept purely as a *leakage diagnostic*:
  `leakage_ratio` $= A_\mathrm{lockin}/A_\mathrm{demod}$ ($\approx1$ when clean,
  $>1$ when the wander leaks).
* An independent **windowed** LPSD single-bin (`speckit`, $A=\sqrt{2\,\mathrm{PS}}$)
  is also leakage-suppressed, so its `method_agreement` with the demodulator is
  small when the tone is well-behaved and flags a genuine problem otherwise.
* **Coherent noise floor:**
  $\sigma_A=\sqrt{\langle|z_\mathrm{off}|^2\rangle/N_\mathrm{eff}}$ with
  $N_\mathrm{eff}=T\cdot 2B$ independent looks — equal to the Cramér–Rao bound
  $\sqrt{S_n(f_0)/T}$ (verified by the Monte-Carlo tests in `tests/`).
* **Physical drift:** the standard deviation of $\mathrm{OPD}(t)$ — genuine
  time-variation, distinct from the noise floor.

The cell below demonstrates the leakage effect and the demodulator *by hand* on
the example file, and shows they agree with the package.

In [ ]:
from scipy.signal import fftconvolve, firwin

phi = (a - b) - np.mean(a - b)
fs = data.fs
t = np.arange(phi.size) / fs
f0 = 100.0

# (1) Rectangular-window lock-in: one linear least-squares solve (leakage-prone).
K, P = 3, 3          # harmonics, polynomial order
cols = []
for k in range(1, K + 1):
    cols += [np.cos(2 * np.pi * k * f0 * t), np.sin(2 * np.pi * k * f0 * t)]
cols += [t ** p for p in range(P + 1)]
M = np.vstack(cols).T
coef, *_ = np.linalg.lstsq(M, phi, rcond=None)
A_lockin = np.hypot(coef[0], coef[1])

# (2) Complex-baseband demodulation "by hand": heterodyne to DC + narrow low-pass.
B = 0.5                                       # one-sided bandwidth [Hz]
zc = phi * np.exp(-2j * np.pi * f0 * t)
fir = firwin(int(6 * fs / B) | 1, B, fs=fs, window=("kaiser", 8.6))
z = 2.0 * fftconvolve(zc, fir, mode="same")   # |z| = zero-to-peak amplitude
A_demod_manual = np.abs(np.mean(z[fir.size:-fir.size]))   # coherent, edges trimmed

print(f"rectangular lock-in     : A = {A_lockin:.6e} cyc  ->  OPD = "
      f"{phase_cycles_to_opd(A_lockin, cfg.freq_dev_peak)*1e3:.4f} mm")
print(f"by-hand demod (B={B} Hz)  : A = {A_demod_manual:.6e} cyc  ->  OPD = "
      f"{phase_cycles_to_opd(A_demod_manual, cfg.freq_dev_peak)*1e3:.4f} mm")

# (3) The package: demodulate() (primary) + windowed single-bin cross-check.
dm = demodulate(phi, fs, f0)
sp = single_bin_amplitude(phi, fs, f0)
print(f"package demod           : A = {dm.amplitude:.6e} cyc  ->  OPD = "
      f"{phase_cycles_to_opd(dm.amplitude, cfg.freq_dev_peak)*1e3:.4f} mm"
      f"   [{dm.mode}, coherence {dm.coherence:.2f}]")
print(f"windowed single-bin     : A = {sp:.6e} cyc  (agreement "
      f"{abs(dm.amplitude - sp) / dm.amplitude:.1e})")
print(f"leakage_ratio           : {A_lockin / dm.amplitude:.2f}  "
      f"(~1 here since this run is clean; it grows when the slow wander leaks)")
print(f"coherent sigma_A        : {dm.amp_unc:.2e} cyc  -> "
      f"{phase_cycles_to_opd(dm.amp_unc, cfg.freq_dev_peak)*1e6:.3f} um")


<a id="single"></a>
## 5. Worked single-file example

`estimate_opd` packages the whole chain into one call and returns an
`OPDResult` with the headline OPD (mode-selected), the coherent noise floor, the
physical drift, the instantaneous $\mathrm{OPD}(t)$, and a set of quality
diagnostics (coherence, `leakage_ratio`, method agreement, ...). The 3-panel
diagnostic figure shows:

1. the differential-phase ASD with the tone marked,
2. the **phase-folded** tone (synchronous averaging over all cycles, which makes
   even a weak tone visible above the broadband noise), and
3. the instantaneous **OPD(t)** from the demodulator, with the headline
   (mode-selected) OPD and its drift band — and the leakage-prone per-segment
   estimates overlaid for comparison.

In [ ]:
r = estimate_opd(example_file, mod_freq=100.0)

print(f"Acquisition : {r.name}")
print(f"Observable  : {r.observable}")
print(f"Tone        : {r.tone_freq:.4f} Hz  (offset {r.tone_freq_offset*1e3:+.2f} mHz, "
      f"SNR {r.tone_snr:.3g})")
print("-" * 60)
print(f"Residual OPD = {r.opd*1e3:.4f} mm   [{r.integration_mode}]")
print(f"   +/- {r.opd_unc*1e6:7.2f} um  (coherent noise floor)")
print(f"   drift {r.opd_drift*1e6:7.2f} um  (physical variation of OPD(t))")
print("-" * 60)
print(f"demod amplitude    : {r.amp_cycles:.4e} cyc   (coherence {r.coherence:.2f})")
print(f"spectral amplitude : {r.amp_cycles_spectral:.4e} cyc")
print(f"lock-in amplitude  : {r.amp_cycles_lockin:.4e} cyc   (leakage x{r.leakage_ratio:.2f})")
print(f"method agreement   : {r.method_agreement:.2e}")
print(f"2nd-harmonic ratio : {r.harmonic_ratio:.2e}")
print(f"equivalent delay   : {r.delay*1e12:.3f} ps")
print(f"per-channel OPD    : "
      + ", ".join(f"{k}={v*1e3:.1f} mm" for k, v in r.channel_opds.items()))

fig = plot_diagnostics(r, data=data)
plt.show()


<a id="full"></a>
## 6. Full FM1 analysis

We now run every FM1 acquisition. Two modulation frequencies were used:
**Day06 = 100 Hz**, **Day09 = 95 Hz** (the peak frequency deviation, and hence
the OPD conversion, is the same for both since the actuator gain is flat). A
small resolver assigns the right frequency per file by its name.

Per-file three-panel diagnostic plots follow the results table.
> Note: this loads ~16 files of ~600 s each, so the cell takes a couple of
> minutes.

In [ ]:
def fm1_freq(path):
    name = os.path.basename(path)
    if "Day09" in name:
        return 95.0
    if "Day06" in name:
        return 100.0
    return None

dataset = estimate_opd_dataset(FM1_DIR, config=cfg, freq_resolver=fm1_freq)
df = dataset.to_dataframe().sort_values("name").reset_index(drop=True)

# Derive experimental categories from the filename for grouping/discussion.
df["day"] = np.where(df["name"].str.contains("Day09"), "Day09", "Day06")
df["anchoring"] = np.select(
    [df["name"].str.contains("Unanchored") | df["name"].str.contains("Released"),
     df["name"].str.contains("Anchored")],
    ["release", "anchored"], default="other")
df["medium"] = np.where(df["name"].str.contains("Vacuum"), "vacuum", "air")
df["rep"] = np.where(df["name"].str.contains("_R2"), "R2", "R1")

cols = ["name", "day", "anchoring", "medium", "rep", "OPD_mm", "OPD_unc_um",
        "OPD_drift_um", "mode", "coherence", "leakage_ratio", "tone_freq_Hz",
        "tone_snr", "method_agreement"]
pd.set_option("display.width", 300)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
df[cols]


### Per-file diagnostics

The cell below renders the three-panel diagnostic figure for **every**
acquisition: differential-phase ASD (tone marked), phase-folded synchronous
average of the tone, and the instantaneous $\mathrm{OPD}(t)$ from the
demodulator (with the mode-selected headline, drift band, coherence, and
`leakage_ratio` annotated).


In [ ]:
for r in dataset:
    fig = plot_diagnostics(r, config=cfg)
    plt.show()


In [ ]:
fig = plot_dataset_summary(dataset)
plt.show()


In [ ]:
# Two very different quantities: the *coherent noise floor* (the statistical
# measurement limit) vs the *physical drift* of OPD(t). We also tally how often
# leakage and phase-incoherence show up per configuration.
g = (df.groupby("anchoring")
       .agg(n=("OPD_mm", "size"),
            OPD_mm_mean=("OPD_mm", "mean"),
            noise_um=("OPD_unc_um", "median"),
            drift_um=("OPD_drift_um", "median"),
            SNR_median=("tone_snr", "median"),
            leakage_med=("leakage_ratio", "median"),
            n_incoherent=("mode", lambda s: int((s == "incoherent").sum())))
       .round(3))
print(g, "\n")

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"anchored": "C0", "release": "C1", "other": "C3"}
for cfgname, sub in df.groupby("anchoring"):
    ax.scatter(sub["OPD_unc_um"], sub["OPD_drift_um"],
               s=40, color=colors.get(cfgname, "C3"), label=cfgname, alpha=0.8)
lims = [1e-1, 1e3]
ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5, label="equal")
ax.set(xscale="log", yscale="log", xlim=lims, ylim=lims,
       xlabel="coherent noise floor [um]",
       ylabel="physical drift of OPD(t) [um]",
       title="Drift, not measurement noise, limits the release runs")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# Repeatability: pair the two runs (R1/R2) of each configuration.
df["config"] = (df["day"] + " " + df["anchoring"] + " " + df["medium"]
                + " " + df["name"].str.extract(r"(NOSHIMS|SHIMMED)").fillna("")[0]
                + np.where(df["name"].str.contains("FLIGHTTORQUED"), " TORQUED", ""))
pivot = df.pivot_table(index="config", columns="rep", values="OPD_mm")
pivot["|R1-R2| [um]"] = (pivot.get("R1") - pivot.get("R2")).abs() * 1e3
pivot.round(4)


<a id="discussion"></a>
## 7. Discussion of results

The residual differential OPD across the FM1 acquisitions lies in the
**~0.09–1.2 mm** range. The headline numbers and their quality flags support a
few clear conclusions (see the tables/plots above):

**1. Spectral leakage — not the modulation — set the earlier numbers on the
drifting runs.** The `leakage_ratio` column makes this explicit: on the
released/torqued acquisitions the naive rectangular lock-in over-reports the OPD
by up to $\sim1.8\times$ (and the old *per-segment* estimates by $\sim10\times$),
purely because the tens-of-cycles low-frequency wander leaks into the tone bin.
The leakage-immune demodulator removes this bias; e.g. the shimmed
flight-torqued vacuum-release R1 drops from $\sim0.16$ mm (leaky) to
$\sim0.09$ mm, matching the independent windowed single-bin.

**2. The measurement itself is not the limitation on the clean runs.** On the
anchored configurations the tone sits far above the noise (coherent SNR
$\sim10^{2}$–$10^{3}$), the demodulator and the windowed single-bin agree to
$\lesssim10^{-2}$, `leakage_ratio` $\approx1$, and the coherent noise floor is
**sub-µm to a few µm**. The frequency-modulation method delivers a precise OPD
readout when the interferometer is mechanically quiet.

**3. Mechanical stability dominates, and some records genuinely wander.** The
noise-floor-vs-drift scatter shows the physical drift of $\mathrm{OPD}(t)$ far
exceeds the statistical floor on the release runs — the OPD is physically
*moving* over the 600 s record (seen directly in the $\mathrm{OPD}(t)$ panel).
On several release/torqued records the tone is not even phase-coherent over the
whole record (`mode = incoherent`, low `coherence`); there the demodulator
falls back to the noise-debiased incoherent amplitude rather than reporting a
falsely precise coherent number.

**4. Repeatability tracks stability, and is best in vacuum.** Anchored-vacuum is
the most repeatable configuration; release cases differ between runs by hundreds
of µm, consistent with their large drift. All run-to-run differences here are
configuration/stability effects, far above the sub-µm statistical precision.

**Practical takeaway:** trust the anchored numbers as precise OPD values; for
the release configurations report the **OPD(t)** time series and its drift (and
heed the `mode`/`coherence` flags), since the residual OPD is genuinely
time-varying there.

<a id="conclusions"></a>
## 8. Conclusions

* A known single-tone **laser-frequency modulation** turns the interferometer
  into a self-calibrating OPD sensor: the residual OPD is
  $\mathrm{OPD} = A_\varphi\, c / \delta\nu_\mathrm{pk}$, read from the amplitude
  of the tone in the **differential** phase — wavelength-independent and immune
  to the large DC interferometer phase.
* A **complex-baseband demodulator** (heterodyne + narrow low-pass) is the
  primary estimator: it is immune to the spectral leakage that biased the naive
  lock-in, yields the instantaneous $\mathrm{OPD}(t)$, and chooses coherent vs
  incoherent integration from a measured coherence. The rectangular lock-in is
  kept only as a `leakage_ratio` diagnostic, and an independent windowed
  single-bin cross-checks the amplitude. The coherent noise floor equals the
  Cramér–Rao bound (verified by the Monte-Carlo tests in `tests/`).
* Applied to FM1, the method measures residual OPDs of ~0.09–1.2 mm with sub-µm
  *statistical* precision on the clean runs; the dominant uncertainty for
  release configurations is **real mechanical drift**, which the
  $\mathrm{OPD}(t)$ diagnostics expose and quantify.

To reproduce the whole analysis outside this notebook:

```bash
python scripts/analyze_fm1.py FM1 results
# or
python -m het_ifo_opd FM1/ --freq-map Day09=95 Day06=100 --out results --plots
```